# Microproyecto 3: Generación condicional de código con LLMs

**Curso:** Modelos Avanzados para el Procesamiento de Lenguaje Natural
**Integrante:** Jean Pierre Betancourt Padilla
**Fecha:** noviembre de 2026

---

## 1. Portada e introducción

### Resumen

En este microproyecto ajustamos dos modelos de lenguaje de familias distintas para generar
funciones en Python a partir de un enunciado en lenguaje natural. Trabajamos con MBPP, un corpus
de problemas básicos de programación que trae, para cada problema, el enunciado, una solución de
referencia y los `assert` con los que se verifica.

Comparamos cuatro configuraciones para medir cuánto aporta el finetuning con LoRA y qué familia
de modelo resuelve mejor la tarea:

| # | Configuración | Modelo | Entrenamiento |
| :-: | :--- | :--- | :--- |
| 1 | Qwen base | Qwen2.5-1.5B | Sin entrenar (zero-shot) |
| 2 | Qwen + LoRA | Qwen2.5-1.5B | LoRA sobre MBPP `train` |
| 3 | SmolLM2 base | SmolLM2-1.7B | Sin entrenar (zero-shot) |
| 4 | SmolLM2 + LoRA | SmolLM2-1.7B | LoRA sobre MBPP `train` |

### Datos

Usamos `google-research-datasets/mbpp` en su configuración `full` con las particiones oficiales:
`train` (~374 problemas) para el finetuning, `validation` (~90) para vigilar el entrenamiento y
`test` (500) reservado solo para la evaluación final.

El prompt que ve el modelo tiene el enunciado del problema y el nombre con la firma de la función
que hay que escribir. No incluye la solución de referencia, ni las clases y helpers que trae el
código, ni los casos de prueba: así el modelo tiene que resolver el problema y no puede copiarse.
Lo que aprende a generar es el código de referencia completo, con sus imports y sus helpers.

### Evaluación

La métrica es **pass@1**: un problema cuenta como correcto solo si el código generado pasa todos
sus `assert`. Evaluamos los 500 problemas de `test` con una sola solución por problema y
decodificación greedy, y usamos el mismo prompt y el mismo `max_new_tokens` en la versión base y
en la finetuneada de cada modelo.

El código generado se ejecuta en un subproceso aparte con un tiempo límite, nunca dentro del
notebook, porque el modelo puede equivocarse y colgar la sesión.

### Entorno

El notebook se escribió en WSL y se ejecuta completo en Google Colab con GPU L4 o A100. En L4 y
A100 usamos bf16; si la sesión cae en una T4, pasamos a fp16 con `GradScaler` sin cambiar código.
Los adaptadores LoRA y los resultados quedan en Google Drive para sobrevivir a una desconexión.

Todo el pipeline es reproducible: fijamos la semilla, la revisión del dataset y las de los dos
modelos, de modo que dos ejecuciones parten de los mismos datos y de los mismos pesos. El notebook
sigue las once secciones que pide el enunciado y se puede ejecutar de principio a fin con
`Run All`.


## 2. Librerías

Dejamos listo el entorno de ejecución antes de tocar los datos: instalamos las librerías,
verificamos la GPU y la precisión mixta, montamos Drive y fijamos la semilla.


Instalamos las versiones de Hugging Face que usamos en el proyecto.


In [ ]:
!pip install -q "transformers>=4.44,<5" "peft>=0.13,<1" "datasets>=2.20,<4" "accelerate>=0.33"


Importamos lo necesario y comprobamos las versiones que quedaron cargadas. `ast` lo usamos para
leer el nombre y la firma de la función, `subprocess` para ejecutar el código generado y `re`
para extraer los bloques de código del texto que devuelve el modelo.


In [ ]:
import ast
import gc
import json
import os
import random
import re
import subprocess
import sys
import tempfile
import time
import warnings

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

import datasets
import peft
import transformers
from datasets import load_dataset
from google.colab import drive
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Entorno cargado")
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("numpy:", np.__version__)


Detectamos la GPU y elegimos la precisión mixta mirando el compute capability: bf16 solo en GPUs
Ampere o superiores (L4, A100), fp16 con `GradScaler` en T4 y en tarjetas más viejas. En la misma
celda dejamos las constantes del proyecto, incluidos los nombres y las revisiones fijas del
dataset y de los dos modelos.


In [ ]:
SEED = 42
MAX_LEN = 512
MAX_NEW_TOKENS = 512
BATCH_SIZE = 16

DATASET_ID = "google-research-datasets/mbpp"
DATASET_REVISION = "4bb6404fdc6cacfda99d4ac4205087b89d32030c"

QWEN_ID = "Qwen/Qwen2.5-1.5B"
QWEN_REVISION = "8faed761d45a263340a0528343f099c05c9a4323"
SMOL_ID = "HuggingFaceTB/SmolLM2-1.7B"
SMOL_REVISION = "effd688a12921b4cc83e3312b6feb579f70f9c71"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    SOPORTA_BF16 = props.major >= 8
else:
    props = None
    SOPORTA_BF16 = False

if SOPORTA_BF16:
    AMP_DTYPE = torch.bfloat16
    NOMBRE_AMP = "bfloat16"
    USE_SCALER = False
else:
    AMP_DTYPE = torch.float16
    NOMBRE_AMP = "float16"
    USE_SCALER = True

if DEVICE == "cuda":
    print("GPU:", props.name)
    print("VRAM:", round(props.total_memory / 1024 ** 3, 1), "GB")
    print("Compute capability:", props.major, ".", props.minor)
else:
    print("No se detectó GPU: el entrenamiento en CPU va a ser muy lento")

print("Precisión mixta:", NOMBRE_AMP)
print("GradScaler:", USE_SCALER)
print("Máximo de tokens por ejemplo:", MAX_LEN)
print("Máximo de tokens nuevos al generar:", MAX_NEW_TOKENS)
print("Batch de evaluación:", BATCH_SIZE)


Montamos Drive y creamos las carpetas donde quedan los adaptadores y los resultados. Si la
sesión de Colab se cae, volvemos a ejecutar y lo encontramos todo ahí.


In [ ]:
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/microproyecto_3"
RESULTADOS_DIR = os.path.join(DRIVE_DIR, "resultados")
CHECKPOINT_QWEN = os.path.join(DRIVE_DIR, "checkpoint_qwen2.5_1.5b_lora")
CHECKPOINT_SMOL = os.path.join(DRIVE_DIR, "checkpoint_smollm2_1.7b_lora")

os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(RESULTADOS_DIR, exist_ok=True)

print("Carpeta de trabajo:", DRIVE_DIR)
print("Resultados:", RESULTADOS_DIR)
print("Adaptador de Qwen:", CHECKPOINT_QWEN)
print("Adaptador de SmolLM2:", CHECKPOINT_SMOL)

if "HF_TOKEN" in os.environ:
    print("HF_TOKEN: configurado")
else:
    print("HF_TOKEN: no está en el entorno, las descargas públicas funcionan igual")


Fijamos la semilla para que los resultados sean reproducibles y definimos la función que libera
la memoria de la GPU entre un modelo y el siguiente.


In [ ]:
def fijar_semilla(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def liberar_vram(modelo=None, optimizador=None):
    if modelo is not None:
        del modelo
    if optimizador is not None:
        del optimizador
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()


fijar_semilla()
print("Semilla:", SEED)


## 3. Carga del dataset y exploración

Cargamos MBPP desde Hugging Face con la revisión fija de la sección anterior y revisamos qué
trae cada problema, cuánto miden los enunciados y las soluciones y cuántos `assert` usa cada uno.
El análisis se hace sobre `train` y `validation`; el split `test` solo se cuenta y se usa para
comprobar que no haya problemas repetidos, porque queda reservado para la evaluación final.


Cargamos el dataset y revisamos los splits y las columnas que trae.


In [ ]:
dataset = load_dataset(DATASET_ID, "full", revision=DATASET_REVISION)

print("Splits del dataset:")
for nombre in dataset:
    cuantos = len(dataset[nombre])
    if cuantos == 1:
        print(" ", nombre, "->", cuantos, "ejemplo")
    else:
        print(" ", nombre, "->", cuantos, "ejemplos")

print()
print("Columnas:", dataset["train"].column_names)


Comparamos los tamaños con lo que dice el enunciado: 500 problemas de `test` y el resto repartido
entre `train` y `validation`.


In [ ]:
TAMANOS_ESPERADOS = {"train": 374, "validation": 90, "test": 500}

for nombre in TAMANOS_ESPERADOS:
    esperado = TAMANOS_ESPERADOS[nombre]
    obtenido = len(dataset[nombre])
    if obtenido == esperado:
        print(nombre, "OK:", obtenido, "ejemplos")
    else:
        print(nombre, "distinto: se esperaban", esperado, "y hay", obtenido)

print()
print("Total de problemas para evaluar:", len(dataset["test"]))


Mostramos dos problemas de `train` completos: el enunciado, la solución de referencia, los casos
de prueba y el código de preparación que traen algunos ejercicios.


In [ ]:
for indice in [0, 1]:
    ejemplo = dataset["train"][indice]
    print("=" * 70)
    print("task_id:", ejemplo["task_id"])
    print()
    print("Enunciado:")
    print(ejemplo["text"])
    print()
    print("Solución de referencia:")
    print(ejemplo["code"])
    print()
    print("Casos de prueba:")
    for linea in ejemplo["test_list"]:
        print(" ", linea)
    print()
    print("Código de preparación:")
    if ejemplo["test_setup_code"].strip() == "":
        print("  (vacío)")
    else:
        print(ejemplo["test_setup_code"])
    print()


Revisamos que ningún `task_id` se repita entre particiones. Si `train` y `test` compartieran
problemas, el pass@1 saldría inflado porque el modelo habría entrenado con ellos.


In [ ]:
ids_train = []
for ejemplo in dataset["train"]:
    ids_train.append(ejemplo["task_id"])

ids_validation = []
for ejemplo in dataset["validation"]:
    ids_validation.append(ejemplo["task_id"])

ids_test = []
for ejemplo in dataset["test"]:
    ids_test.append(ejemplo["task_id"])

repetidos_train_validation = 0
for ident in ids_train:
    if ident in ids_validation:
        repetidos_train_validation = repetidos_train_validation + 1

repetidos_train_test = 0
for ident in ids_train:
    if ident in ids_test:
        repetidos_train_test = repetidos_train_test + 1

repetidos_validation_test = 0
for ident in ids_validation:
    if ident in ids_test:
        repetidos_validation_test = repetidos_validation_test + 1


def imprimir_split(nombre, ids):
    if len(ids) == 1:
        print(f"{nombre:<11}{len(ids):>4} problema, del {min(ids)} al {max(ids)}")
    else:
        print(f"{nombre:<11}{len(ids):>4} problemas, del {min(ids)} al {max(ids)}")


imprimir_split("train", ids_train)
imprimir_split("validation", ids_validation)
imprimir_split("test", ids_test)
print()
print("Repetidos entre train y validation:", repetidos_train_validation)
print("Repetidos entre train y test:", repetidos_train_test)
print("Repetidos entre validation y test:", repetidos_validation_test)


Medimos los enunciados y las soluciones de `train` y `validation` en caracteres. Si los dos splits
se parecen, lo que aprendamos mirando `train` también aplica a `validation`.


In [ ]:
def promedio(valores):
    return sum(valores) / len(valores)


def percentil(valores, fraccion):
    ordenados = sorted(valores)
    posicion = int(len(ordenados) * fraccion)
    return ordenados[posicion]


largos_texto_train = []
largos_codigo_train = []
palabras_train = []
for ejemplo in dataset["train"]:
    largos_texto_train.append(len(ejemplo["text"]))
    largos_codigo_train.append(len(ejemplo["code"]))
    palabras_train.append(len(ejemplo["text"].split()))

largos_texto_validation = []
largos_codigo_validation = []
for ejemplo in dataset["validation"]:
    largos_texto_validation.append(len(ejemplo["text"]))
    largos_codigo_validation.append(len(ejemplo["code"]))

print(f"{'split':<12}{'n':>5}{'texto_prom':>12}{'texto_p95':>11}{'codigo_prom':>13}{'codigo_p95':>12}")

for nombre in ["train", "validation"]:
    if nombre == "train":
        textos = largos_texto_train
        codigos = largos_codigo_train
    else:
        textos = largos_texto_validation
        codigos = largos_codigo_validation
    texto_promedio = round(promedio(textos), 1)
    texto_p95 = percentil(textos, 0.95)
    codigo_promedio = round(promedio(codigos), 1)
    codigo_p95 = percentil(codigos, 0.95)
    print(f"{nombre:<12}{len(textos):>5}{texto_promedio:>12}{texto_p95:>11}{codigo_promedio:>13}{codigo_p95:>12}")


Miramos con más detalle los largos de `train`, que es el split con el que se entrena. Todavía no
usamos un tokenizador porque cada modelo tiene el suyo y eso lo vemos en la sección siguiente.


In [ ]:
print("Enunciado en caracteres")
print("  mínimo:", min(largos_texto_train))
print("  promedio:", round(promedio(largos_texto_train), 1))
print("  mediana:", percentil(largos_texto_train, 0.50))
print("  p95:", percentil(largos_texto_train, 0.95))
print("  máximo:", max(largos_texto_train))
print()
print("Enunciado en palabras")
print("  mínimo:", min(palabras_train))
print("  promedio:", round(promedio(palabras_train), 1))
print("  mediana:", percentil(palabras_train, 0.50))
print("  p95:", percentil(palabras_train, 0.95))
print("  máximo:", max(palabras_train))
print()
print("Solución en caracteres")
print("  mínimo:", min(largos_codigo_train))
print("  promedio:", round(promedio(largos_codigo_train), 1))
print("  mediana:", percentil(largos_codigo_train, 0.50))
print("  p95:", percentil(largos_codigo_train, 0.95))
print("  máximo:", max(largos_codigo_train))


Comparamos los dos splits con diagramas de caja. La caja muestra la mediana y el rango donde está
la mitad central de los datos, y los puntos sueltos son problemas atípicos.


In [ ]:
plt.figure(figsize=(10, 3))

plt.subplot(1, 2, 1)
plt.boxplot([largos_texto_train, largos_texto_validation])
plt.xticks([1, 2], ["train", "validation"])
plt.title("Enunciados")
plt.ylabel("Caracteres")

plt.subplot(1, 2, 2)
plt.boxplot([largos_codigo_train, largos_codigo_validation])
plt.xticks([1, 2], ["train", "validation"])
plt.title("Soluciones")
plt.ylabel("Caracteres")

plt.tight_layout()
plt.show()


Dibujamos la curva acumulada de las longitudes: para cada valor del eje x muestra qué porcentaje
de los problemas mide eso o menos. Sirve para decidir cuánto contexto necesitamos.


In [ ]:
def acumulada(valores):
    ordenados = sorted(valores)
    x = ordenados
    y = []
    for i in range(len(ordenados)):
        y.append((i + 1) / len(ordenados))
    return x, y


x_train, y_train = acumulada(largos_texto_train)
x_val, y_val = acumulada(largos_texto_validation)

plt.figure(figsize=(10, 3))

plt.subplot(1, 2, 1)
plt.plot(x_train, y_train, label="train")
plt.plot(x_val, y_val, linestyle="--", label="validation")
plt.axvline(percentil(largos_texto_train, 0.95), linestyle=":", color="gray")
plt.title("Enunciados")
plt.xlabel("Caracteres")
plt.ylabel("Proporción acumulada")
plt.legend()

x_train, y_train = acumulada(largos_codigo_train)
x_val, y_val = acumulada(largos_codigo_validation)

plt.subplot(1, 2, 2)
plt.plot(x_train, y_train, label="train")
plt.plot(x_val, y_val, linestyle="--", label="validation")
plt.axvline(percentil(largos_codigo_train, 0.95), linestyle=":", color="gray")
plt.title("Soluciones")
plt.xlabel("Caracteres")
plt.legend()

plt.tight_layout()
plt.show()

print("p95 del enunciado en train:", percentil(largos_texto_train, 0.95), "caracteres")
print("p95 de la solución en train:", percentil(largos_codigo_train, 0.95), "caracteres")


Revisamos qué tan completas son las soluciones de referencia. Si varias traen imports, clases o
funciones auxiliares, el modelo tiene que aprender a generarlas y no basta con el cuerpo de una
sola función.


In [ ]:
def contar_estructura(particion):
    conteos = {"una función": 0, "varias funciones": 0, "con imports": 0, "con clases": 0, "sin leer": 0}
    for ejemplo in particion:
        try:
            arbol = ast.parse(ejemplo["code"])
        except SyntaxError:
            conteos["sin leer"] = conteos["sin leer"] + 1
            continue

        imports = 0
        clases = 0
        funciones = 0
        for nodo in arbol.body:
            if isinstance(nodo, ast.Import):
                imports = imports + 1
            if isinstance(nodo, ast.ImportFrom):
                imports = imports + 1
            if isinstance(nodo, ast.ClassDef):
                clases = clases + 1
            if isinstance(nodo, ast.FunctionDef):
                funciones = funciones + 1

        if funciones == 1:
            conteos["una función"] = conteos["una función"] + 1
        if funciones > 1:
            conteos["varias funciones"] = conteos["varias funciones"] + 1
        if imports > 0:
            conteos["con imports"] = conteos["con imports"] + 1
        if clases > 0:
            conteos["con clases"] = conteos["con clases"] + 1
    return conteos


conteos_train = contar_estructura(dataset["train"])
conteos_validation = contar_estructura(dataset["validation"])

print("train:", conteos_train)
print("validation:", conteos_validation)

categorias = ["una función", "varias funciones", "con imports", "con clases"]

porcentaje_train = []
porcentaje_validation = []
for categoria in categorias:
    porcentaje_train.append(100 * conteos_train[categoria] / len(dataset["train"]))
    porcentaje_validation.append(100 * conteos_validation[categoria] / len(dataset["validation"]))

plt.figure(figsize=(8, 3.5))
plt.barh(categorias, porcentaje_train, label="train")
plt.barh(categorias, porcentaje_validation, label="validation", alpha=0.6)
plt.xlabel("Porcentaje de problemas")
plt.title("Estructura del código de referencia")
plt.legend()
plt.tight_layout()
plt.show()


Buscamos los casos extremos de `train`: el enunciado más largo, la solución más larga y el único
problema que trae `test_setup_code`. Los extremos son los que pueden quedar truncados o romper la
ejecución de los `assert`.


In [ ]:
indice_texto = largos_texto_train.index(max(largos_texto_train))
indice_codigo = largos_codigo_train.index(max(largos_codigo_train))

print("Enunciado más largo de train")
print("  task_id:", dataset["train"][indice_texto]["task_id"])
print("  caracteres:", largos_texto_train[indice_texto])
print(" ", dataset["train"][indice_texto]["text"])
print()
print("Solución más larga de train")
print("  task_id:", dataset["train"][indice_codigo]["task_id"])
print("  caracteres:", largos_codigo_train[indice_codigo])
print("  líneas:", len(dataset["train"][indice_codigo]["code"].splitlines()))

indice_setup = None
for i in range(len(dataset["train"])):
    if dataset["train"][i]["test_setup_code"].strip() != "":
        indice_setup = i
        break

print()
if indice_setup is None:
    print("Ningún problema de train trae test_setup_code")
else:
    print("Problema con test_setup_code")
    print("  task_id:", dataset["train"][indice_setup]["task_id"])
    print(dataset["train"][indice_setup]["test_setup_code"])


Revisamos la columna `challenge_test_list`, que trae pruebas adicionales, y miramos un ejemplo del
split `prompt`, que no usamos ni para entrenar ni para evaluar.


In [ ]:
con_challenge = 0
for ejemplo in dataset["train"]:
    if len(ejemplo["challenge_test_list"]) > 0:
        con_challenge = con_challenge + 1

print("Problemas de train con challenge_test_list:", con_challenge, "de", len(dataset["train"]))

ejemplo_prompt = dataset["prompt"][0]
print()
print("Un ejemplo del split prompt:")
print("task_id:", ejemplo_prompt["task_id"])
print()
print(ejemplo_prompt["text"])
print()
print(ejemplo_prompt["code"])


Mostramos un problema de `validation` para confirmar que tiene el mismo formato que `train`.


In [ ]:
ejemplo_validacion = dataset["validation"][0]
print("task_id:", ejemplo_validacion["task_id"])
print()
print(ejemplo_validacion["text"])
print()
print(ejemplo_validacion["code"])
print()
print("Primer caso de prueba:", ejemplo_validacion["test_list"][0])


Revisamos cuántos `assert` trae cada problema y en cuántos aparece `test_setup_code`, porque eso
cambia la forma de ejecutar la verificación: el código de preparación va después de la solución
generada y antes de los `assert`.


In [ ]:
for nombre in ["train", "validation"]:
    conteo_asserts = {}
    con_setup = 0
    for ejemplo in dataset[nombre]:
        cuantos = len(ejemplo["test_list"])
        if cuantos in conteo_asserts:
            conteo_asserts[cuantos] = conteo_asserts[cuantos] + 1
        else:
            conteo_asserts[cuantos] = 1
        if ejemplo["test_setup_code"].strip() != "":
            con_setup = con_setup + 1

    print(nombre)
    for cuantos in sorted(conteo_asserts):
        if cuantos == 1:
            etiqueta = "assert"
        else:
            etiqueta = "asserts"
        if conteo_asserts[cuantos] == 1:
            print(" ", cuantos, etiqueta, "->", conteo_asserts[cuantos], "problema")
        else:
            print(" ", cuantos, etiqueta, "->", conteo_asserts[cuantos], "problemas")
    print("  problemas con test_setup_code:", con_setup)
    print()


### Lo que nos deja la exploración

`train` va del `task_id` 601 al 974 y `validation` del 511 al 600, sin ningún identificador
repetido entre particiones. Eso descarta que el modelo haya visto durante el entrenamiento algún
problema de los que se usan para monitorear, y deja `test` (del 11 al 510) limpio para el pass@1
final.

Los dos splits se parecen: el enunciado promedia 78 caracteres (unas 14 palabras) y la solución
178, con un p95 de 113 y 412 caracteres. El 95 % de las soluciones cabe en menos de 412
caracteres, así que un límite de 512 tokens sobra, y la solución más larga ronda las 900 letras.

Casi todas las soluciones son una sola función, pero 87 de las 374 traen imports y unas pocas
agregan clases o funciones auxiliares. Por eso el target es el código completo y no solo el
cuerpo: si entrenáramos solo con el cuerpo, el modelo nunca aprendería a escribir esas piezas.

Los problemas de MBPP no dicen cómo se llama la función, pero los `assert` la llaman por su
nombre. Por eso el prompt lleva el enunciado y el nombre con la firma de la función. La solución
de referencia y los casos de prueba se quedan fuera del prompt, porque si aparecen el modelo
podría copiarse o memorizar los `assert` en vez de resolver el problema.

Además, los 374 problemas de `train` traen exactamente 3 `assert` y solo uno usa
`test_setup_code`, así que la verificación es uniforme y el caso especial es raro pero hay que
manejarlo: ese código de preparación se ejecuta después de la solución generada y antes de los
`assert`.


## 4. Construcción de prompts y preprocesamiento

Pendiente: formato de instrucción, tokenización y máscara de labels con -100.


## 5. Particiones (train / validation / test)

Pendiente: asignación de roles y verificación de que test no se usa antes de la evaluación.


## 6. Implementación de la evaluación pass@1

Pendiente: generación por lotes, extracción de código y ejecución segura de asserts.


## 7. Evaluación de los modelos base (zero-shot)

Pendiente: pass@1 de Qwen2.5-1.5B y SmolLM2-1.7B originales sobre los 500 de test.


## 8. Finetuning con LoRA

Pendiente: configuración de LoRA, bucle de entrenamiento y seguimiento en validation.


## 9. Evaluación de los modelos finetuneados

Pendiente: pass@1 de cada modelo con su adaptador LoRA sobre el mismo test.


## 10. Comparación y resultados

Pendiente: tabla de las cuatro configuraciones, gráficas y ejemplos cualitativos.


## 11. Conclusiones

Pendiente: discusión de resultados, seguimiento del entrenamiento y limitaciones.
